# Qblox QRM-RF readout test

This notebook targets one qubit on QRM-RF slot 7, IO port 1, at cluster address `192.168.0.2`. It does not configure or use drive channels. Install Qibolab and a Qblox integration/driver dependency set compatible with this repository (the installation docs refer to `qibolab-qblox`).

**Hardware safety:** `parameters.json` contains zero-valued placeholders. Before enabling hardware, replace them with your calibrated readout frequency, QRM-RF LO frequency and power, acquisition delay/smearing, and readout pulse amplitude/duration. Confirm the cabling and that the cluster is safe to operate. `RUN_HARDWARE` is disabled by default.

In [ ]:
import os
from pathlib import Path

from qibolab import (
    Acquisition,
    AcquisitionType,
    Gaussian,
    Parameter,
    Pulse,
    PulseSequence,
    Readout,
    Sweeper,
    create_platform,
)

if "QIBOLAB_PLATFORMS" not in os.environ:
    candidates = [Path.cwd(), *Path.cwd().parents]
    platforms_dir = next(
        (
            parent / "platforms"
            for parent in candidates
            if (parent / "platforms" / "qblox_readout" / "platform.py").is_file()
        ),
        next(
            (
                parent
                for parent in candidates
                if parent.name == "platforms"
                and (parent / "qblox_readout" / "platform.py").is_file()
            ),
            None,
        ),
    )
    if platforms_dir is None:
        raise RuntimeError(
            "Could not find platforms/qblox_readout/platform.py above the current "
            "working directory. Set QIBOLAB_PLATFORMS to the absolute path of the "
            "platforms directory."
        )
    os.environ["QIBOLAB_PLATFORMS"] = str(platforms_dir.resolve())

platform = create_platform("qblox_readout")
qubit = platform.qubits["q0"]
print(f"Platform: {platform.name}")
print(f"Readout probe: {qubit.probe} -> {platform.channels[qubit.probe]}")
print(f"Acquisition: {qubit.acquisition} -> {platform.channels[qubit.acquisition]}")
print("Configs:")
for name in (qubit.probe, qubit.acquisition, "qrm_rf/o1/lo"):
    print(f"  {name}: {platform.parameters.configs[name]}")

## Review calibration and pulse values

Edit `platforms/qblox_readout/parameters.json` with the values from your calibration/runcard. Then enter the calibrated readout pulse duration and normalized amplitude below. Keep `CALIBRATION_REVIEWED` false until those values and the wiring have been checked.

In [ ]:
READOUT_DURATION_NS = 500  # calibrated integration/probe duration, in ns
READOUT_AMPLITUDE = 0.001  # calibrated normalized amplitude, between -1 and 1
N_SHOTS = 1000
CALIBRATION_REVIEWED = True  # set True only after checking all hardware values

In [ ]:
assert CALIBRATION_REVIEWED, "Review the hardware wiring and calibration first."
assert READOUT_DURATION_NS is not None and READOUT_DURATION_NS > 0
assert READOUT_AMPLITUDE is not None and 0 < abs(READOUT_AMPLITUDE) <= 1

probe_config = platform.parameters.configs[qubit.probe]
acquisition_config = platform.parameters.configs[qubit.acquisition]
lo_config = platform.parameters.configs["qrm_rf/o1/lo"]
assert probe_config.frequency > 0, "Set the calibrated probe frequency."
assert lo_config.frequency > 0, "Set the calibrated QRM-RF LO frequency."
print(
    "Calibration values are populated; acquisition delay/smearing:",
    acquisition_config.delay,
    acquisition_config.smearing,
)

In [ ]:
probe = Pulse(
    duration=READOUT_DURATION_NS,
    amplitude=READOUT_AMPLITUDE,
    relative_phase=0.0,
    envelope=Gaussian(rel_sigma=0.2),
)
sequence = PulseSequence(
    [
        (
            qubit.acquisition,
            Readout(
                probe=probe,
                acquisition=Acquisition(duration=READOUT_DURATION_NS),
            ),
        )
    ]
)
print(f"Prepared one readout operation ({READOUT_DURATION_NS} ns).")

In [ ]:
RUN_HARDWARE = True  # Change to True only after reviewing the warnings above.

if not RUN_HARDWARE:
    print("Hardware execution skipped. Set RUN_HARDWARE = True to connect and acquire.")
else:
    assert CALIBRATION_REVIEWED, "Review the hardware wiring and calibration first."
    platform.connect()
    try:
        results = platform.execute(
            [sequence],
            nshots=N_SHOTS,
            acquisition_type=AcquisitionType.INTEGRATION,
        )
    finally:
        platform.disconnect()

    readout_id = sequence.acquisitions[0][1].id
    print(f"Integrated results for readout {readout_id}:")
    print(results[readout_id])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Integration acquisition returns one (I, Q) pair per shot.
iq = np.asarray(results[readout_id], dtype=float)
if iq.ndim != 2 or iq.shape[1] != 2 or iq.shape[0] == 0:
    raise ValueError(f"Expected a non-empty (nshots, 2) I/Q array; got {iq.shape}.")

I, Q = iq.T
fig, (ax_iq, ax_hist) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_iq.scatter(I, Q, s=24, alpha=0.75)
ax_iq.set(title="Integrated readout: I/Q cloud", xlabel="I", ylabel="Q")
ax_iq.grid(True, alpha=0.25)

ax_hist.hist(I, bins="auto", alpha=0.65, label="I")
ax_hist.hist(Q, bins="auto", alpha=0.65, label="Q")
ax_hist.set(
    title="Integrated readout distributions", xlabel="Integrated value", ylabel="Shots"
)
ax_hist.legend()
ax_hist.grid(True, alpha=0.25)

fig.suptitle(f"QRM-RF readout, {len(iq)} shots")
fig.tight_layout()
plt.show()
print(f"I: mean={I.mean():.6g}, std={I.std():.6g}")
print(f"Q: mean={Q.mean():.6g}, std={Q.std():.6g}")

In [ ]:
center = platform.parameters.configs[qubit.probe].frequency

# Start with a modest scan around the current probe frequency.
frequencies = np.linspace(center - 400e6, center + 400e6, 1001)

sweeper = Sweeper(
    parameter=Parameter.frequency,
    values=frequencies,
    channels=[qubit.probe],
)

platform.connect()
try:
    results = platform.execute(
        [sequence],
        [[sweeper]],
        nshots=N_SHOTS,
        acquisition_type=AcquisitionType.INTEGRATION,
    )
finally:
    platform.disconnect()

readout_id = sequence.acquisitions[0][1].id
iq_by_shot = np.asarray(results[readout_id])
mean_iq = iq_by_shot.mean(axis=0)  # average shots at each frequency
response = mean_iq[:, 0] + 1j * mean_iq[:, 1]

fig, (ax_mag, ax_iq) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
ax_mag.plot(frequencies / 1e9, np.abs(response), ".-")
ax_mag.set_ylabel("|I + iQ|")
ax_mag.set_title("Resonator spectroscopy")
ax_mag.grid(True)

ax_iq.plot(frequencies / 1e9, mean_iq[:, 0], label="I")
ax_iq.plot(frequencies / 1e9, mean_iq[:, 1], label="Q")
ax_iq.set_xlabel("Probe frequency (GHz)")
ax_iq.set_ylabel("Integrated response")
ax_iq.legend()
ax_iq.grid(True)
plt.tight_layout()
plt.show()